# 4 深度学习计算

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

## 1. 模型构造

### 1.1 继承 nn.Module 构造模型

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [3]:
X = torch.rand(2, 20)
net = MLP()
net(X)

tensor([[-0.0285, -0.1068, -0.0979,  0.0196, -0.0579,  0.0218, -0.0011,  0.2195,
          0.0126, -0.1588]], grad_fn=<AddmmBackward0>)

### 1.2 使用 Sequential 构造模型

In [4]:
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 30)
)
net(X)

tensor([[ 0.0463,  0.0337,  0.0297,  0.0043, -0.0176, -0.0311,  0.0244,  0.0248,
          0.0160,  0.0107,  0.0332,  0.0457,  0.0236, -0.0032,  0.0202,  0.0080,
          0.0021, -0.0260,  0.0241, -0.0032, -0.0057,  0.0071, -0.0289, -0.0227,
          0.0093, -0.0014,  0.0127, -0.0035, -0.0279, -0.0281]],
       grad_fn=<AddmmBackward0>)

### 1.3 使用 ModuleList 和 ModuleDict

In [5]:
class MyModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.choices = nn.ModuleDict({
            'relu': nn.ReLU(),
            'linear': nn.Linear(20, 10)
        })
        self.layers = nn.ModuleList([
            nn.Linear(10, 10) for _ in range(3)
        ])

    def forward(self, X):
        X = self.choices['linear'](X)
        for layer in self.layers:
            X = F.relu(layer(X))
        return X

model = MyModule()
print('ModuleDict keys:', model.choices.keys())
print('ModuleList length:', len(model.layers))
print('output shape:', model(X).shape)

ModuleDict keys: odict_keys(['relu', 'linear'])
ModuleList length: 3
output shape: torch.Size([2, 10])


## 2. 参数管理

### 2.1 访问参数：parameters() 和 named_parameters()

In [6]:
net = MLP()
print('=== named_parameters() ===')
for name, param in net.named_parameters():
    print(f'{name}: {param.shape}')

print()
print('=== 第一层权重 ===')
print(net.hidden.weight)

print()
print('=== state_dict() keys ===')
print(', '.join(net.state_dict().keys()))

=== named_parameters() ===
hidden.weight: torch.Size([256, 20])
hidden.bias: torch.Size([256])
out.weight: torch.Size([10, 256])
out.bias: torch.Size([10])

=== 第一层权重 ===
tensor([[ 0.1092, -0.0021, -0.2389,  ...,  0.0706, -0.0428,  0.1353],
        [ 0.2202, -0.0555, -0.1861,  ...,  0.0683,  0.1357,  0.1325],
        [-0.0642,  0.1152,  0.2201,  ..., -0.1773,  0.0234, -0.1272],
        ...,
        [ 0.1451, -0.0565,  0.1922,  ...,  0.1507,  0.1331,  0.1941],
        [-0.0169, -0.1495,  0.0925,  ...,  0.1809,  0.0964,  0.1482],
        [-0.1646,  0.0609, -0.1149,  ...,  0.0125, -0.0284,  0.1262]])

=== state_dict() keys ===
hidden.weight, hidden.bias, out.weight, out.bias


### 2.2 初始化参数

In [7]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

net = MLP()
net.apply(init_weights)
print(f'初始化后 hidden 权重均值: {net.hidden.weight.mean().item():.4f}')
print(f'初始化后 hidden 权重标准差: {net.hidden.weight.std().item():.4f}')
print(f'初始化后 out 权重均值: {net.out.weight.mean().item():.4f}')
print(f'初始化后 out 权重标准差: {net.out.weight.std().item():.4f}')

初始化后 hidden 权重均值: 0.0016
初始化后 hidden 权重标准差: 0.0116
初始化后 out 权重均值: -0.0015
初始化后 out 权重标准差: 0.1578


### 2.3 共享参数

In [8]:
shared = nn.Linear(8, 5)
net = nn.Sequential(
    nn.Linear(8, 8),
    nn.ReLU(),
    shared,
    nn.ReLU(),
    shared
)
X = torch.rand(2, 8)
print('shared.weight is net[2].weight:', net[2].weight is net[4].weight)
print('前向传播输出:', net(X))

shared.weight is net[0].weight: True
前向传播输出: tensor([[ 0.1196, -0.1183, -0.0376, -0.0498,  0.0960],
        [ 0.1353, -0.1402, -0.0434, -0.0593,  0.1090]],
       grad_fn=<AddmmBackward0>)


## 3. 自定义层

### 3.1 自定义层（不含参数）

In [9]:
class CenteredLayer(nn.Module):
    def forward(self, X):
        return X - X.mean()

layer = CenteredLayer()
layer(torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))

tensor([[0.9000, 0.6000, 0.3000],
        [0.8000, 0.7000, 0.6000]])

### 3.2 自定义层（含参数）

In [10]:
class MyLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, X):
        return X @ self.weight + self.bias

linear = MyLinear(4, 4)
linear(torch.rand(2, 4))

tensor([[ 0.3822,  0.4573,  1.2216,  1.9726],
        [ 0.2403,  0.5032,  1.4212,  1.5302]], grad_fn=<AddmmBackward0>)

## 4. 读写文件

### 4.1 张量的保存与加载

In [11]:
x = torch.arange(6).reshape(2, 3)
y = torch.tensor([10, 20, 30])
print('原始张量 x:', x)
print('原始张量 y:', y)

torch.save(x, 'x.pt')
torch.save([x, y], 'xy.pt')

x_loaded = torch.load('x.pt')
xy_loaded = torch.load('xy.pt')
print()
print('加载后的张量 x_loaded:', x_loaded)
print('加载后的张量 y_loaded:', xy_loaded[1])

原始张量 x: tensor([[1, 2, 3],
        [4, 5, 6]])
原始张量 y: tensor([10, 20, 30])

加载后的张量 x_loaded: tensor([[1, 2, 3],
        [4, 5, 6]])
加载后的张量 y_loaded: tensor([10, 20, 30])


### 4.2 模型参数的保存与加载

In [12]:
net1 = MLP()
init_weights(net1)
mean_before = net1.hidden.weight.mean().item()

torch.save(net1.state_dict(), 'mlp.pt')

net2 = MLP()
net2.load_state_dict(torch.load('mlp.pt'))
mean_after = net2.hidden.weight.mean().item()

print(f'保存前: net1 第一层权重均值 {mean_before:.4f}')
print(f'保存后: net2 第一层权重均值 {mean_after:.4f}')
same = all((p1 == p2).all() for p1, p2 in zip(net1.parameters(), net2.parameters()))
print(f'参数一致: {same}')

保存前: net1 第一层权重均值 -0.0015
保存后: net2 第一层权重均值 -0.0015
参数一致: True


### 4.3 保存完整模型

In [13]:
torch.save(net1, 'mlp_full.pt')
net_loaded = torch.load('mlp_full.pt')
out = net_loaded(torch.rand(1, 20))
print('完整模型保存并加载成功')
print('加载后模型输出:', out)

完整模型保存并加载成功
加载后模型输出: tensor([[ 0.0166,  0.0682, -0.0161, -0.0296,  0.0496,  0.0451, -0.0440, -0.0301,
         -0.0391,  0.0200]], grad_fn=<AddmmBackward0>)


## 5. GPU

### 5.1 检查 GPU 可用性

In [14]:
print('CUDA 可用:', torch.cuda.is_available())
print('GPU 数量:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU 名称:', torch.cuda.get_device_name(0))
print('当前设备:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

CUDA 可用: True
GPU 数量: 1
GPU 名称: NVIDIA GeForce RTX 4090
当前设备: cuda:0


### 5.2 张量在 CPU 与 GPU 间移动

In [15]:
x = torch.tensor([1, 2, 3])
print('CPU 张量设备:', x.device)

if torch.cuda.is_available():
    device = torch.device('cuda')
    x_gpu = x.to(device)
    print('转移到 GPU 后设备:', x_gpu.device)
    x_gpu2 = x.cuda()
    print('用 .cuda() 转移到 GPU 后设备:', x_gpu2.device)
    x_cpu = x_gpu.cpu()
    print('移回 CPU 后设备:', x_cpu.device)

CPU 张量设备: cpu
转移到 GPU 后设备: cuda:0
用 .cuda() 转移到 GPU 后设备: cuda:0
移回 CPU 后设备: cpu


### 5.3 GPU 上的计算

In [16]:
if torch.cuda.is_available():
    a = torch.rand(2, 3).cuda()
    b = torch.rand(2, 3).cuda()
    c = a + b
    c

tensor([[ 2.7614,  2.8026,  2.9890],
        [ 3.8090,  3.7535,  4.0616]], device='cuda:0')

### 5.4 模型移动到 GPU

In [17]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    net = MLP()
    net.to(device)
    print('模型参数设备:', next(net.parameters()).device)
    X = torch.rand(1, 20).to(device)
    output = net(X)
    print('输出设备:', output.device)
    print('输出:', output)

模型参数设备: cuda:0
输出设备: cuda:0
输出: tensor([[ 0.0707,  0.1152, -0.0427,  0.0080, -0.0242,  0.0513,  0.0272, -0.0039,
          0.0171, -0.1383]],
       grad_fn=<AddmmBackward0>)
